# Homogeneous Baseline Experiments

The goal of this notebook is to train and evaluate different Graph Neural Network architectures (GAT, GraphSAGE, GCN) on the pure, noiseless **formation assignment** task.

We want to find out which architecture has the best capacity to learn the combinatorial Hungarian algorithm before we introduce realistic physical noise.


In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn.models import GAT, GCN, GraphSAGE
from torch_geometric.data import InMemoryDataset
import pandas as pd
from pathlib import Path
import numpy as np

# We assume standard dimensions based on your data collection
IN_CHANNELS = 13  # Local lin vel(3) + local ang vel(3) + obs features(4) + formation one-hot(3)
MAX_SLOTS = 20    # Max classes (slots) to predict

c:\Users\aimen\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Generate Noiseless Dataset

We generate a clean dataset specifically for the `formation_assignment_homo` task.
_(Note: We copy the `GeneratedSplitDataset` wrapper to load the generated `.pt` files)._


In [ ]:
# Make sure to run your data_collection_notebook.ipynb once, or import generate_dataset here.
# For this experiment, you would set:
#  generate_dataset(
#      dataset_name="noiseless_baseline",
#      task_type="formation_assignment_homo",
#      noisy_sensors=False,v
#      environmental_wind=False,
#      dynamic_formation=False,
#      inject_failures=False,
#      num_episodes=500  # Good size for baseline
# )


In [19]:
class GeneratedSplitDataset(InMemoryDataset):
    def __init__(self, split_path: str | Path):
        self.split_path = Path(split_path)
        payload = torch.load(self.split_path, weights_only=False)
        super().__init__(root="")
        self.data, self.slices = payload["data"], payload["slices"]

# Load datasets (Replace with your actual path once generated)
datasets_dir = Path.cwd().parent / "datasets"
train_dataset = GeneratedSplitDataset(datasets_dir / "noiseless_baseline_mixed_formations_train.pt")
val_dataset = GeneratedSplitDataset(datasets_dir / "noiseless_baseline_mixed_formations_val.pt")
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

## 2. Define Model Architectures

We initialize the pre-built PyG models. We'll store them in a dictionary to easily loop through them for experiments.


In [20]:
models = {
    "GCN": GCN(in_channels=IN_CHANNELS, hidden_channels=64, num_layers=3, out_channels=MAX_SLOTS, dropout=0.0),
    "GraphSAGE": GraphSAGE(in_channels=IN_CHANNELS, hidden_channels=64, num_layers=3, out_channels=MAX_SLOTS, dropout=0.0),
    "GAT": GAT(in_channels=IN_CHANNELS, hidden_channels=64, num_layers=3, out_channels=MAX_SLOTS, v2=True, dropout=0.0)
}

## 3. Training & Evaluation Logic

Here we define the functions to train the model and evaluate the domain-specific metrics (Conflicts, Accuracies).


In [21]:
def train_model(model, train_loader, epochs=10, lr=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            # Forward pass
            out = model(batch.x, batch.edge_index)
            
            # Cross entropy for multiclass classification
            loss = F.cross_entropy(out, batch.target.long().view(-1))
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * batch.num_graphs
            
        # print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss / len(train_loader.dataset):.4f}")
    return model

def evaluate_model(model_name, model, loader):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    
    correct_nodes = 0
    total_nodes = 0
    
    total_graphs = 0
    conflict_free_graphs = 0
    
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index)
            preds = out.argmax(dim=-1)
            targets = batch.target.long().view(-1)
            
            # 1. Node-level Accuracy
            correct_nodes += (preds == targets).sum().item()
            total_nodes += targets.size(0)
            
            # 2. Graph-level Conflict Checking
            # Since graphs are batched, we must check for duplicate slot assignments PER GRAPH
            for i in range(batch.num_graphs):
                # Get the predictions for the i-th graph in the batch
                graph_mask = batch.batch == i
                graph_preds = preds[graph_mask]
                
                total_graphs += 1
                # If the number of unique slot predictions equals the number of drones, there are NO collisions
                if len(torch.unique(graph_preds)) == len(graph_preds):
                    conflict_free_graphs += 1

    node_acc = correct_nodes / total_nodes
    conflict_rate = 1.0 - (conflict_free_graphs / total_graphs)
    perfect_assignment_rate = conflict_free_graphs / total_graphs
    
    return {
        "Model": model_name,
        "Node Accuracy": f"{node_acc * 100:.2f}%",
        "Conflict Rate": f"{conflict_rate * 100:.2f}%",
        "Perfect Assignment": f"{perfect_assignment_rate * 100:.2f}%"
    }

## 4. Run Experiments

Uncomment and run this block once your data loaders are initialized. It will test all 3 architectures and display a neat table.


In [22]:
results = []
for name, model in models.items():
    print(f"Training {name}...")
    trained_model = train_model(model, train_loader, epochs=20)
    metrics = evaluate_model(name, trained_model, val_loader)
    results.append(metrics)

df_results = pd.DataFrame(results)
display(df_results)

Training GCN...


KeyboardInterrupt: 